
# MVP Análise de Dados e Boas Práticas

**Nome:** Joao Paulo de Moraes Candido Guido

**Matrícula:** 4052025002311

**Dataset:** [Netflix Movies and TV Shows](https://www.kaggle.com/code/shivamb/netflix-shows-and-movies-exploratory-analysis)

---

# 1. Descrição do Problema

## Contexto e Objetivo
Este projeto propõe uma investigação analítica sobre o ecossistema de conteúdo da Netflix, utilizando técnicas de **Análise Exploratória de Dados (EDA)**. O foco central é compreender a transição estratégica da plataforma de um repositório de licenciamento para uma potência de produção global. Através do processamento deste dataset, buscamos identificar padrões que fundamentam as decisões de negócio da empresa até o ano de 2021.

## Hipóteses de Trabalho
Para nortear esta análise, estabelecemos as seguintes premissas:
1. **Saturação vs. Expansão:** A Netflix atingiu um platô de crescimento nos EUA, forçando uma diversificação massiva para mercados emergentes (como Índia e Coreia do Sul).
2. **Mudança de Formato:** Há uma transição clara no investimento de Filmes (Movies) para Séries (TV Shows), visando maior retenção de usuários (*engagement*).
3. **Segmentação Etária:** A estratégia de conteúdo é predominantemente focada no público adulto (TV-MA), consolidando a marca como um serviço de entretenimento maduro.

## Tipo de Problema
Este é um problema de **Análise Exploratória de Dados (EDA)** e **Aprendizado Não Supervisionado**, focado em descoberta de conhecimento (KDD), com o objetivo de identificar padrões e tendências no catálogo da Netflix.

## Dicionário de Dados
- ***show_id:*** Identificador único (Primary Key) de cada título no catálogo.
- ***type:*** Categoria do conteúdo: Movie (Filme) ou TV Show (Série).
- ***title:*** Título oficial da obra.
- ***director:*** Diretor(es) responsáveis pela produção (pode conter múltiplos nomes ou valores ausentes).
- ***cast:*** Lista de atores e atrizes que compõem o elenco principal.
- ***country:*** País ou países onde a obra foi produzida (útil para análise de abrangência geográfica).
- ***date_added:*** Data em que o título foi disponibilizado na plataforma Netflix.
- ***release_year:*** Ano de lançamento original da produção.
- ***rating:*** Classificação indicativa/etária do conteúdo (ex: TV-MA, PG-13, L), não se referindo a notas de usuários.
- ***duration:*** Medida de tempo: em minutos para filmes e em temporadas (Seasons) para séries.
- ***listed_in:*** Gêneros e subcategorias nos quais a obra está classificada (ex: Documentários, Comédias).
- ***description:*** Sinopse ou breve resumo do enredo da obra.


# Importação das Bibliotecas e Carga dos dados

Esta seção consolida todas as importações de bibliotecas necessárias para a análise, visualização e pré-processamento dos dados, bem como o carregamento inicial do dataset Netflix

In [1]:
# instalando a dependencia do plotly para construir melhores graficos
!pip install dash

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import io
import numpy as np
import plotly as py
import pandas as pd
import seaborn as sns
import datetime as dt
import matplotlib.colors
import plotly.express as px
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.figure_factory as ff
from plotly.subplots import make_subplots

from PIL import Image
from wordcloud import WordCloud, STOPWORDS

from dash import Dash, html, dcc, callback, Output, Input

from collections import Counter
from itertools import combinations

from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans, AffinityPropagation
from sklearn.preprocessing import LabelEncoder, StandardScaler


### Montar a palheta de cores da netflix para usar nos graficos

[Logo da Marca](https://brand.netflix.com/en/assets/logos)

In [3]:
TEMPLATE_STYLE = "plotly_white"
COLORS = {'Netflix_Red': '#E50914', 'Netflix_Red_':'#b20710', 'Dark_Grey': '#221F1F', 'Light_Grey': '#F5F5F1', 'Grey': '#4a4a4a'}

### Configuração do pandas para não quebrar linhas nas exibições

In [4]:
# 1. Mostrar todas as colunas (não truncar com "...")
pd.set_option('display.max_columns', None)

# 2. Aumentar a largura máxima exibida por coluna (para textos longos)
pd.set_option('display.max_colwidth', None)

# 3. Evitar que o pandas quebre a linha se o terminal/notebook for estreito
# Define uma largura muito grande ou ilimitada para o display
pd.set_option('display.width', 1000)

## Leitura do dataset e print das informações do proprio

In [5]:
# Carga do dataset
def load_and_inspect(path):
    try:
        df = pd.read_csv(path)
        print(f"[INFO] Dataset carregado: {df.shape[0]} instâncias e {df.shape[1]} atributos.")
        return df
    except Exception as e:
        print(f"[ERRO] Falha ao carregar arquivo: {e}")
        return None

df_netflix = load_and_inspect('./netflix_titles.csv')

def get_data_overview(df: pd.DataFrame) -> pd.DataFrame:
    """
    Objetivo: Entender a informação disponível (Checklist: Estatísticas Descritivas).
    """
    print("--- Visão Geral do Dataset ---")
    print("Tipos de Dados:")
    print(df.dtypes)
    print("\nResumo de Valores Ausentes:")
    print(df.isnull().sum())
    return df.describe(include='all')

get_data_overview(df_netflix)

[INFO] Dataset carregado: 8807 instâncias e 12 atributos.
--- Visão Geral do Dataset ---
Tipos de Dados:
show_id         object
type            object
title           object
director        object
cast            object
country         object
date_added      object
release_year     int64
rating          object
duration        object
listed_in       object
description     object
dtype: object

Resumo de Valores Ausentes:
show_id            0
type               0
title              0
director        2634
cast             825
country          831
date_added        10
release_year       0
rating             4
duration           3
listed_in          0
description        0
dtype: int64


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
count,8807,8807,8807,6173,7982,7976,8797,8807.000000,8803,8804,8807,8807
unique,8807,2,8807,4528,7692,748,1767,NaN,17,220,514,8775
top,s8807,Movie,Zubaan,Rajiv Chilaka,David Attenborough,United States,"January 1, 2020",NaN,TV-MA,1 Season,"Dramas, International Movies","Paranormal activity at a lush, abandoned property alarms a group eager to redevelop the site, but the eerie events may not be as unearthly as they think."
freq,1,6131,1,19,19,2818,109,NaN,3207,1793,362,4
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2014.180198,NaN,NaN,NaN,NaN
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.819312,NaN,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1925.000000,NaN,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2013.000000,NaN,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2017.000000,NaN,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2019.000000,NaN,NaN,NaN,NaN


# 2. Pré-processamento e Limpeza de Dados

Nesta etapa, realizamos o tratamento de valores nulos e a engenharia de recursos para transformar atributos brutos em dados analisáveis. Seguindo as boas práticas, não descartaremos linhas com diretores nulos, mas os rotularemos como "Não Especificado".


## Tratamento de Dados e Qualidade (Data Cleaning)
A integridade da análise depende da qualidade dos dados brutos. Identifiquei lacunas críticas nas colunas `director`, `cast` e `country`. 
*   **Estratégia de Imputação:** Para as variáveis categóricas `director` e `cast`, optei pelo preenchimento com o rótulo **'Unknown'** (Desconhecido), evitando a exclusão de 30% do dataset, o que enviesaria as análises temporais. 
*   **Tratamento Geográfico:** Para a coluna `country`, apliquei a **imputação pela moda**, assumindo que títulos sem origem declarada em uma plataforma americana possuem alta probabilidade de serem produções dos EUA. Esta escolha visa preservar o maior número de registros, minimizando a perda de informação.
*   **Engenharia de Atributos:** Realizei a extração de novas variáveis a partir de `date_added` para permitir análises sazonais (mês e ano de adição), essenciais para validar a hipótese de expansão temporal.


In [6]:
def preprocess_pipeline(df):
    """
    Executa operações de limpeza, transformação, engenharia de atributos
    e separação de métricas de duração por categoria.
    """

    # 1. Preparação Inicial
    df_clean = df.copy()

    # Preenchimento de valores ausentes (Categorias fixas)
    df_clean['cast'] = df_clean['cast'].fillna('Unknown')
    df_clean['rating'] = df_clean['rating'].fillna('Unknown')
    df_clean['director'] = df_clean['director'].fillna('Unknown')

    # Preenchimento de valores ausentes (Moda)
    df_clean['country'] = df_clean['country'].fillna(df_clean['country'].mode()[0])
    df_clean['duration'] = df_clean['duration'].fillna(df_clean['duration'].mode()[0])
    df_clean['date_added'] = df_clean['date_added'].fillna(df_clean['date_added'].mode()[0])

    # 2. Engenharia de Datas
    df_clean['date_added'] = pd.to_datetime(df_clean['date_added'], format='mixed')
    df_clean['added_year'] = df_clean['date_added'].dt.year.astype(int)
    df_clean['added_month'] = df_clean['date_added'].dt.month.astype(int)
    df_clean['added_month_name'] = df_clean['date_added'].dt.month_name()

    # Criando o 'Release Lag' (Tempo entre lançamento e entrada na plataforma)
    df_clean['release_lag'] = df_clean['added_year'] - df_clean['release_year']
    
    # Limpeza de inconsistências lógicas
    df_clean = df_clean[df_clean['release_lag'] >= 0]

    # 3. Tratamento e Separação de Duração
    # Extrai apenas os números da coluna 'duration'
    raw_duration = df_clean['duration'].str.extract(r'(\d+)').astype(float)

    # Cria máscaras booleanas para separação
    movie_mask = df_clean['type'] == 'Movie'
    show_mask = df_clean['type'] == 'TV Show'

    # Separação em colunas distintas (Minutos vs Temporadas)
    df_clean['duration_movie_min'] = np.where(movie_mask, raw_duration[0], np.nan)
    df_clean['duration_show_season'] = np.where(show_mask, raw_duration[0], np.nan)

    # 4. Mapeamento de Classificação Etária
    rating_map = {
        'TV-Y': 'Kids', 'TV-Y7': 'Kids', 'TV-Y7-FV': 'Kids', 'G': 'Kids', 'TV-G': 'Kids',
        'PG': 'Teens', 'TV-PG': 'Teens', 'PG-13': 'Teens', 'TV-14': 'Teens',
        'R': 'Adults', 'TV-MA': 'Adults', 'NC-17': 'Adults',
        'NR': 'Unknown', 'UR': 'Unknown'
    }
    df_clean['rating_age'] = df_clean['rating'].map(rating_map).fillna('Unknown')

    # 5. Normalização e Encoding
    scaler = StandardScaler()

    # Padronização isolada para Filmes (Minutos)
    df_clean.loc[movie_mask, 'duration_movie_scaled'] = scaler.fit_transform(
        df_clean.loc[movie_mask, ['duration_movie_min']]
    )

    # Padronização isolada para Séries (Temporadas)
    df_clean.loc[show_mask, 'duration_show_scaled'] = scaler.fit_transform(
        df_clean.loc[show_mask, ['duration_show_season']]
    )

    # 6. Encoding Categórico
    le = LabelEncoder()
    df_clean['type_encoded'] = le.fit_transform(df_clean['type'])
    
    # Extração do país principal (First Country)
    df_clean['first_country'] = df_clean['country'].apply(lambda x: x.split(',')[0]).replace({
        'United States': 'USA',
        'United Kingdom': 'UK',
        'South Korea': 'S. Korea'
    })
    
    return df_clean

In [7]:
df_netflix = preprocess_pipeline(df_netflix)

In [8]:
df_netflix.info()

<class 'pandas.core.frame.DataFrame'>
Index: 8793 entries, 0 to 8806
Data columns (total 23 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   show_id                8793 non-null   object        
 1   type                   8793 non-null   object        
 2   title                  8793 non-null   object        
 3   director               8793 non-null   object        
 4   cast                   8793 non-null   object        
 5   country                8793 non-null   object        
 6   date_added             8793 non-null   datetime64[ns]
 7   release_year           8793 non-null   int64         
 8   rating                 8793 non-null   object        
 9   duration               8793 non-null   object        
 10  listed_in              8793 non-null   object        
 11  description            8793 non-null   object        
 12  added_year             8793 non-null   int64         
 13  added_mo

# 3. Análise de Dados e Visualizações

Nesta etapa de Análise de Dados Exploratória (EDA) sobre o dataset da natflix, visamos entender a distribuição, as relações e as características das variáveis, o que é crucial para as etapas subsequentes de pré-processamento e modelagem.

## 3.1. Estatísticas Descritivas

Abaixo, observamos o resumo estatístico dos dados numéricos.

In [9]:
df_netflix.describe()

,date_added,release_year,added_year,added_month,release_lag,duration_movie_min,duration_show_season,duration_movie_scaled,duration_show_scaled,type_encoded
count,8793,8793.000000,8793.000000,8793.000000,8793.000000,6129.000000,2664.000000,6.129000e+03,2.664000e+03,8793.000000
mean,2019-05-17 21:48:00.245649920,2014.172182,2018.874559,6.645286,4.702377,99.532876,1.760886,-2.463539e-16,7.901587e-17,0.302968
min,2008-01-01 00:00:00,1925.000000,2008.000000,1.000000,0.000000,1.000000,1.000000,-3.473253e+00,-4.814186e-01,0.000000
25%,2018-04-06 00:00:00,2013.000000,2018.000000,4.000000,0.000000,87.000000,1.000000,-4.417800e-01,-4.814186e-01,0.000000
50%,2019-07-04 00:00:00,2017.000000,2019.000000,7.000000,1.000000,98.000000,1.000000,-5.403342e-02,-4.814186e-01,0.000000
75%,2020-08-19 00:00:00,2019.000000,2020.000000,10.000000,5.000000,114.000000,2.000000,5.099616e-01,1.512894e-01,1.000000
max,2021-09-25 00:00:00,2021.000000,2021.000000,12.000000,93.000000,312.000000,17.000000,7.489400e+00,9.641911e+00,1.000000
std,NaN,8.823846,1.572868,3.438870,8.788448,28.371359,1.580804,1.000082e+00,1.000188e+00,0.459568


### **Análise:**
1. Temporalidade e Recência (Release & Added)

    * Aceleração de Conteúdo: O catálogo é predominantemente moderno. Embora existam títulos desde 1925, a mediana (50%) do ano de lançamento é 2017. Isso indica que metade do acervo foi lançada nos últimos 9 anos.

    * Pico de Adições: A média de entrada na plataforma (added_year) é 2018, com o terceiro quartil (75%) em 2020. Isso demonstra que o grande volume de expansão do catálogo ocorreu entre 2018 e 2021.

    * O "Gargalo" de Lançamento (release_lag): Em média, um título leva 4,7 anos desde o seu lançamento original até ser adicionado à plataforma. No entanto, o desvio padrão de 8,78 é alto, sugerindo que a plataforma mistura lançamentos quase simultâneos com clássicos muito antigos (o lag máximo chega a 93 anos).

2. Métricas de Duração (Movies vs. Shows)

    * Filmes (Standard de 1h40): A duração média dos filmes (duration_movie_min) é de 99,5 minutos. O desvio padrão de 28,37 mostra uma consistência relativa, mas o intervalo é vasto: desde produções curtíssimas de 1 minuto (provavelmente curtas ou testes) até épicos de 312 minutos (5,2 horas).

    * Séries (Fidelidade Curta): A média de temporadas (duration_show_season) é baixa, de apenas 1,76. Pelo menos 75% das séries no catálogo possuem apenas 2 temporadas ou menos. Isso aponta para um volume alto de minisséries ou produções canceladas precocemente, embora o valor máximo de 17 temporadas indique a presença de franquias consolidadas.

3. Distribuição de Formato (type_encoded)

    * Predomínio Binário: O valor médio de 0.30 na coluna codificada (onde provavelmente 0 = Movie e 1 = TV Show) confirma estatisticamente o que vimos no gráfico de rosca: o catálogo tem aproximadamente 70% de filmes e 30% de séries.

4. Qualidade dos Dados e Outliers

    * Limpeza Necessária: Note que o count para duration_movie_min (6.129) e duration_show_season (2.664) é diferente do total de linhas (8.793). Isso é um indicador técnico de que essas colunas foram filtradas ou tratadas separadamente por tipo, já que um Filme não possui temporadas e uma Série não tem uma duração única em minutos.

    * Outliers de Duração: Os valores máximos (312 min para filmes e 17 temporadas para séries) estão muito distantes do terceiro quartil (75%), o que os caracteriza como outliers que podem distorcer a média, tornando a mediana (50%) um indicador mais confiável para o planejamento de conteúdo.

**Resumo para Negócio**: Os dados descrevem um catálogo de consumo rápido e atual, focado em filmes de cerca de 1h40 e séries curtas, com um esforço massivo de licenciamento e adição de títulos ocorrido principalmente a partir de 2018.

## 3.2. Contexto Histórico: A Evolução da Netflix

Antes de nos aprofundarmos nas analises do dataset, é fundamental entender os marcos temporais da empresa. Isso nos ajudará a validar se os dados refletem eventos reais, como a transição para o streaming e a expansão global.


In [10]:
# Dados
dates = ["1997<br>Fundação", "1998<br>Serviço Postal", "2003<br>Abertura de Capital",
         "2007<br>Serviço de Streaming", "2016<br>Expansão Global", "2021<br>Fenômeno Mundial"]
x_main = [1, 2, 4, 5.3, 8, 9.5]

sub_x = [1.5, 3, 5, 6.5, 7.3]
sub_years = ["1998", "2000", "2006", "2010", "2012"]
sub_text = ["Lançamento do<br>Netflix.com", "Início das<br>Recomendações",
            "Bilionésima<br>Entrega de DVD", "Lançamento<br>no Canadá", "Lançamento no<br>Reino Unido"]
levels = [0.4, -0.5, 0.45, -0.5, 0.7]

fig = go.Figure()

# Linha horizontal central
fig.add_shape(type="line", x0=0.6, y0=0, x1=10, y1=0, line=dict(color=COLORS['Grey'], width=2))

# Pontos Principais
fig.add_trace(go.Scatter(
    x=x_main, y=[0]*6, mode='markers+text',
    text=dates, textposition="bottom center",
    marker=dict(color=COLORS['Light_Grey'], size=15, line=dict(color=COLORS['Grey'], width=3)),
    name="Marcos Principais"
))

# Eventos Secundários (Hastes e Textos)
for x, y, year, txt in zip(sub_x, levels, sub_years, sub_text):
    color = COLORS['Netflix_Red_'] if "Reino Unido" in txt else COLORS['Grey']

    # Linha vertical
    fig.add_shape(type="line", x0=x, y0=0, x1=x, y1=y, line=dict(color=color, width=1))

    # Texto do Evento
    fig.add_trace(go.Scatter(
        x=[x], y=[y], mode="text",
        text=f"<b>{year}</b><br>{txt}",
        textposition="top center" if y > 0 else "bottom center",
        textfont=dict(color=color, size=11),
        showlegend=False
    ))

# Layout
fig.update_layout(
    title=dict(text="<b>Netflix ao longo dos anos</b><br><sup>De aluguel de DVDs a uma audiência global</sup>", x=0.5),
    plot_bgcolor='white',
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False, range=[0, 10.5]),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False, range=[-1.2, 1.5]),
    margin=dict(l=20, r=20, t=80, b=20),
    showlegend=False
)

fig.show()

**Analise do Gráfico:**
A linha do tempo destaca a transformação radical da Netflix: de uma empresa de logística de DVDs (1997-2006) para uma gigante de tecnologia e streaming (2007 em diante). Este gráfico é crucial para interpretarmos o dataset, pois explica por que a maioria dos dados de "data_added" se concentra após 2016 (ano da expansão global). Um ponto de atenção para a análise posterior é que, embora a fundação seja em 1997, o volume massivo de títulos no catálogo atual reflete a estratégia pós-2010 de internacionalização e produção original.

## 3.3 Conteudos

Agora que entendemos um pouco da história e evolução da netflix, vamos entender o que eles tem a oferecer.

In [11]:
# Pergunta: Qual a distribuição de frequência das classes (Movie vs TV Show)?
type_counts = df_netflix['type'].value_counts().reset_index()
type_counts.columns = ['Tipo', 'Total']

fig1 = px.pie(
    type_counts, values='Total', names='Tipo', 
    title='Distribuição de Conteúdo: Filmes vs Séries',
    hole=0.5, 
    color_discrete_sequence=[COLORS['Netflix_Red'], COLORS['Dark_Grey']]
) # Cores Netflix, mas em Donut Chart interativo

fig1.update_layout(
    title=dict(text="<b>Distribuição de Conteúdo: Filmes vs Séries</b><br>", x=0.5),
    plot_bgcolor='white',
    margin=dict(l=20, r=20, t=80, b=20),
    showlegend=False
)

fig1.update_traces(
    textposition='inside', 
    textinfo='percent+label'
)
fig1.show()

#### ANÁLISE Distribuição de Conteúdo: Filmes vs Séries: 

O catálogo é massivamente composto por filmes, que representam 69,7% do total. Isso indica que, para cada 10 títulos disponíveis, aproximadamente 7 são filmes.

As séries ocupam 30,3% do gráfico. Embora em menor quantidade numérica, é importante considerar que o "volume de horas" de conteúdo das séries pode ser muito maior do que o dos filmes, dependendo do número de temporadas e episódios.

Existe uma proporção de aproximadamente 2,3 para 1. Ou seja, há mais que o dobro de filmes em relação a séries.

Ponto de Atenção: Para um modelo de recomendação, o desbalanceamento entre as classes pode exigir técnicas de amostragem ou pesos diferenciados.

In [12]:
# --- 3. RATING AGE POR TIPO ---
df_rating = df_netflix.groupby(['rating_age', 'type']).size().unstack().fillna(0)

fig6 = go.Figure()
for t, color in zip(['Movie', 'TV Show'], [COLORS['Netflix_Red'], COLORS['Dark_Grey']]):
    fig6.add_trace(
        go.Bar(
            name=t, x=df_rating.index, y=df_rating[t], marker_color=color,
            text=df_rating[t], textposition='outside', textfont=dict(family='serif', color=COLORS['Dark_Grey'], size=12)
        )
    )

fig6.update_layout(barmode='group', title="<b>Distribuição de Classificação Etária (Rating)</b>", font_family="serif", plot_bgcolor='white')

fig6.show()

#### ANÁLISE "Distribuição de Classificação Etária (Rating)": 

1. Concentração de Volume: Adults e Teens

* O catálogo é massivamente focado nos públicos Adults (3.999 títulos) e Teens (3.798 títulos). A diferença de volume total entre esses dois grandes pilares é mínima, o que reforça que a plataforma busca atender prioritariamente ao público jovem e adulto.

2. O Equilíbrio Único no Segmento Infantil

    O segmento Kids é o único onde o número de TV Shows (464) supera o de Movies (442), ainda que por uma margem pequena.

    * Insight: Isso reflete o comportamento de consumo infantil, que é muito baseado em repetição e séries (desenhos animados, programas educativos), demandando um volume maior de conteúdo episódico do que obras únicas.

3. A Estratégia de "Filmes para Massa"

    Nas categorias Adults e Teens, a quantidade de filmes é quase o triplo da quantidade de séries:

    * Adults: 2.860 filmes vs 1.139 séries.
    * Teens: 2.744 filmes vs 1.054 séries.
    * Isso confirma que o volume bruto que sustenta o catálogo para esses públicos é cinematográfico, usando séries como um conteúdo mais "premium" ou selecionado.

In [13]:
# --- 1. PREPARAÇÃO DOS DADOS (Tratamento de Alta Cardinalidade e Multivaloração) ---
# Criamos uma cópia para não fragmentar o dataframe principal
df_exploded = df_netflix.copy()

# Separamos os gêneros que estão agrupados por vírgula e expandimos o DataFrame
df_exploded['genre'] = df_exploded['listed_in'].str.split(', ')
df_exploded = df_exploded.explode('genre')

# Identificamos os 10 gêneros mais frequentes para evitar poluição visual (Sparsity)
top_10_genres = df_exploded['genre'].value_counts().nlargest(10).index

# Filtramos o dataframe explodido para conter apenas os top gêneros
df_top_genres = df_exploded[df_exploded['genre'].isin(top_10_genres)]

# --- 2. CRIAÇÃO DA MATRIZ (Crosstab) ---
# IMPORTANTE: Ambos os argumentos devem vir de 'df_top_genres' para evitar erro de índice duplicado
cross_tab = pd.crosstab(
    df_top_genres['genre'], 
    df_top_genres['rating_age'], 
    normalize='index' # Normalizamos por linha para ver a % de público dentro de cada gênero
)

# --- 3. VISUALIZAÇÃO INTERATIVA COM PLOTLY ---
fig20 = px.imshow(
    cross_tab, 
    text_auto=".2f", 
    aspect="auto",
    title="<b>Matriz de Correlação: Gênero vs. Classificação Etária</b><br><sup>Proporção de audiência por categoria de conteúdo</sup>",
    color_continuous_scale='Reds',
    labels=dict(x="Classificação Etária (Rating)", y="Gênero", color="Proporção")
)

fig20.update_layout(
    template=TEMPLATE_STYLE,
    xaxis_title="Público-Alvo",
    yaxis_title="Gêneros (Top 10)",
    margin=dict(t=100)
)

fig20.show()

#### ANÁLISE Matriz de Correlação: Gênero vs. Classificação Etária:
Esta Matriz de Correlação (Heatmap) foca na relação entre os Top 10 Gêneros e o Público-Alvo (Adults, Kids, Teens, Unknown). Diferente do gráfico anterior, que mostrava o volume total, este revela o perfil demográfico de cada categoria.

Aqui estão os insights mais relevantes:
1. O Público "Teens" é o Motor de Diversidade

    O grupo Teens (Adolescentes) é o público mais onipresente em quase todos os gêneros. Eles detêm a maior proporção em categorias variadas como:

    * Romantic Movies (0.63): O ponto mais alto de concentração para este público.
    * Comedies (0.56) e Documentaries (0.51).
    * Curiosamente, eles dividem o mercado de Action & Adventure (0.50) quase que igualmente com os adultos.

2. Segmentação Clara para Adultos

    O público Adults domina gêneros mais sérios ou de nicho artístico:

    * Independent Movies (0.71): É o gênero com a maior concentração específica de um único público em toda a matriz. Quase 3/4 dos filmes independentes são voltados para adultos.
    * TV Dramas (0.57) e Dramas (0.50) também mostram uma inclinação forte para este público.

3. Exclusividade e Isolamento de "Kids"

    O gênero Children & Family Movies é o único que apresenta uma correlação quase nula com os outros públicos (0.00 para Adults), concentrando-se em:

    * Kids (0.52) e Teens (0.48).
    * Isso indica que o conteúdo para crianças raramente é "crossover" para o público estritamente adulto neste conjunto de dados.

4. Gêneros Transversais (Equilíbrio)

    Alguns gêneros mostram uma distribuição muito equilibrada entre adultos e adolescentes, sugerindo um apelo mais universal:

    * Action & Adventure: Quase um empate perfeito (0.49 Adults vs 0.50 Teens).
    * International TV Shows: Também apresenta equilíbrio (0.53 Adults vs 0.45 Teens), indicando que produções estrangeiras atraem ambas as faixas etárias de forma similar.

5. Dados "Unknown"

    A coluna Unknown é irrelevante estatisticamente (0.01 a 0.03), o que é um bom sinal para a qualidade dos seus dados, indicando que quase todo o catálogo está corretamente classificado por faixa etária.

In [14]:
# --- 1. PREPARAÇÃO DOS DADOS ---
# Filtramos apenas os 5 principais países
countries_list = df_netflix['first_country'].unique()[:5].tolist()
# df_top3 = df_netflix[df_netflix['first_country'].isin(countries_list)]

# Agrupamos por ano de adição e país para contar os títulos
# Nota: 'release_year' é o ano de lançamento, 'added_year' seria o ano que entrou na Netflix.
# Vou usar 'release_year' para ver a evolução da produção disponível.
df_evolucao = df_netflix.groupby(['release_year', 'first_country']).size().reset_index(name='count')

# --- 2. CRIAÇÃO DO GRÁFICO ---
fig3 = go.Figure()

for country in countries_list:
    df_c = df_evolucao[df_evolucao['first_country'] == country]

    fig3.add_trace(go.Scatter(
        x=df_c['release_year'],
        y=df_c['count'],
        mode='lines',
        name=country,
        # line=dict(color=colors[country], width=3),
        hovertemplate="<b>" + country + "</b><br>Ano: %{x}<br>Títulos: %{y}<extra></extra>"
    ))

# --- 3. AJUSTES DE LAYOUT ---
fig3.update_layout(
    title=dict(
        text="<b>Evolução da Produção de Conteúdo</b><br><sup>Quantidade de títulos por ano de lançamento</sup>",
        x=0.05,
        font=dict(family="serif", size=22, color=COLORS['Dark_Grey'])
    ),
    xaxis=dict(title="Ano de Lançamento", showgrid=False, range=[2007, 2021] # Focando no período desde o inicio do serviço de streaming
    ),
    yaxis=dict(title="Quantidade de Títulos", showgrid=True, gridcolor=COLORS['Light_Grey']),
    plot_bgcolor='white',
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    height=500,
    margin=dict(t=100, b=50, l=50, r=50)
)

fig3.show()

#### ANÁLISE Evolução da Produção de Conteúdo:

O gráfico oferece uma perspectiva histórica sobre a origem geográfica e o ritmo de crescimento dos títulos ao longo dos anos.

Aqui estão os insights fundamentais desse novo conjunto de dados:
1. Hegemonia e o "Pico" dos EUA

* Os Estados Unidos (linha azul) dominam o catálogo em volume absoluto de forma isolada. É possível observar um crescimento acelerado a partir de 2014, atingindo o ápice por volta de 2018-2019 com quase 500 títulos lançados por ano. No entanto, há uma queda acentuada após 2020, o que pode refletir o impacto da pandemia na produção ou mudanças nas estratégias de licenciamento.
2. A Ascensão da Índia

* A Índia (linha verde) consolidou-se como o segundo maior polo de produção do catálogo. Seu crescimento é consistente e ganha fôlego a partir de 2016, mantendo-se significativamente acima de mercados tradicionais como o Reino Unido (UK). Isso reforça o peso do conteúdo internacional na plataforma.
3. Estabilidade do Reino Unido (UK)

* O Reino Unido (linha roxa) apresenta uma trajetória de crescimento gradual e estável, com um pico de produção entre 2016 e 2018. Embora não tenha o mesmo volume da Índia ou EUA, é um fornecedor constante de conteúdo para o portfólio.
4. Mercados Emergentes e Estáticos

* Países como África do Sul (linha vermelha) e Alemanha (linha laranja) mantêm uma presença tímida e quase linear no gráfico, com volumes baixos (abaixo de 50 títulos anuais), indicando que são fontes de conteúdo de nicho ou muito pontuais para este catálogo específico.

In [15]:
# --- 1. COMPARATIVO: USA VS OUTROS PAÍSES ---
# Criamos uma categoria binária para destacar os EUA
df_netflix['is_usa'] = df_netflix['first_country'].apply(lambda x: 'USA' if x == 'USA' else 'Outros Países')
usa_vs_others = df_netflix['is_usa'].value_counts()

fig4 = go.Figure(data=[go.Pie(
    labels=usa_vs_others.index,
    values=usa_vs_others.values,
    hole=.5,
    marker=dict(colors=[COLORS['Dark_Grey'], '#b20710']),
    textinfo='percent+label'
)])
fig4.update_layout(title="<b>Volume de Conteúdo: USA vs. Resto do Mundo</b>", font_family="serif")
fig4.show()

#### ANÁLISE "Volume de Conteúdo: USA vs. Resto do Mundo":

Esse grafico é a perspectiva de globalização do catálogo.

Aqui estão os insights fundamentais:
1. Equilíbrio Geográfico

    * Apesar da forte presença histórica dos Estados Unidos (que vimos no gráfico de evolução), o catálogo como um todo é majoritariamente internacional. Os "Outros Países" somam 54,1%, superando os 45,9% dos EUA.

    * Insight: Isso caracteriza uma plataforma com forte apelo global, que não depende exclusivamente de Hollywood para sustentar seu inventário.

2. Diversidade de Origem

    * O fato de mais da metade do conteúdo vir de fora dos EUA sugere que a plataforma investe pesado em licenciamento e produções locais (como vimos com a ascensão da Índia e a constância do Reino Unido). Isso é uma estratégia chave para expansão em mercados internacionais, atendendo a públicos que buscam representatividade cultural.

In [16]:
# --- 5. CORRELAÇÃO DE GÊNEROS (LISTED_IN) ---
# Vamos explodir os gêneros para ver quais aparecem juntos com mais frequência
# Filtramos apenas filmes para o exemplo de correlação
genres_list = df_netflix[df_netflix['type'] == 'Movie']['listed_in'].str.split(', ').tolist()
all_combinations = []

for genres in genres_list:
    all_combinations.extend(combinations(sorted(genres), 2))

common_pairs = Counter(all_combinations).most_common(15)
labels_pairs = [f"{p[0][0]} & {p[0][1]}" for p in common_pairs]
values_pairs = [p[1] for p in common_pairs]

fig8 = go.Figure(go.Bar(
    x=values_pairs,
    y=labels_pairs,
    orientation='h',
    marker=dict(
        color=values_pairs,
        colorscale='Reds', # Gradiente para destacar a frequência
        line=dict(color=COLORS['Dark_Grey'], width=1)
    ),
    hovertemplate="Combinação: %{y}<br>Frequência: %{x}<extra></extra>"
))

# --- 3. AJUSTES DE LAYOUT ---
fig8.update_layout(
    title="<b>Top 15 Combinações de Gêneros Mais Comuns (Filmes)</b><br><sup>Análise de adjacência de categorias</sup>",
    xaxis_title="Quantidade de Títulos",
    yaxis_title="Pares de Gêneros",
    yaxis_autorange="reversed", # Maior valor no topo
    plot_bgcolor='white',
    font_family="serif",
    height=600,
    margin=dict(t=100, l=150)
)

fig8.show()

In [17]:
# Filtramos apenas as Series para o exemplo de correlação
genres_list = df_netflix[df_netflix['type'] == 'TV Show']['listed_in'].str.split(', ').tolist()
all_combinations = []
for genres in genres_list:
    all_combinations.extend(combinations(sorted(genres), 2))

common_pairs = Counter(all_combinations).most_common(15)
labels_pairs = [f"{p[0][0]} & {p[0][1]}" for p in common_pairs]
values_pairs = [p[1] for p in common_pairs]

fig9 = go.Figure(go.Bar(
    x=values_pairs,
    y=labels_pairs,
    orientation='h',
    marker=dict(
        color=values_pairs,
        colorscale='Reds', # Gradiente para destacar a frequência
        line=dict(color=COLORS['Dark_Grey'], width=1)
    ),
    hovertemplate="Combinação: %{y}<br>Frequência: %{x}<extra></extra>"
))

# --- 3. AJUSTES DE LAYOUT ---
fig9.update_layout(
    title="<b>Top 15 Combinações de Gêneros Mais Comuns (Series)</b><br><sup>Análise de adjacência de categorias</sup>",
    xaxis_title="Quantidade de Títulos",
    yaxis_title="Pares de Gêneros",
    yaxis_autorange="reversed", # Maior valor no topo
    plot_bgcolor='white',
    font_family="serif",
    height=600,
    margin=dict(t=100, l=150)
)

fig9.show()

#### ANÁLISE Top 15 Combinações de Gêneros (Filmes e Series):

Estes dois gráficos de barras acima revelam o "DNA" do conteúdo disponível, mostrando como os títulos são classificados para atrair audiências específicas.

Aqui estão os insights mais relevantes:

1. A Espinha Dorsal: Conteúdo Internacional e Drama

* Tanto para filmes quanto para séries, a combinação International + Drama é a líder absoluta e isolada.

    * Em Filmes, a dupla "Dramas & International Movies" atinge quase 1.500 títulos, sendo quase o dobro da segunda colocada "Comedies & International Movies" com pouco mais de 800 títulos.

    * Em Séries, "International TV Shows & TV Dramas" domina com pouco mais de 500 títulos, enquanto a segunda colocada "International TV Shows & Romantic TV Show" segue com mais de 300 titulos.

    * Insight: O drama internacional é o "porto seguro" do catálogo, sendo o gênero com maior investimento e volume global.

2. O Diferencial das Séries: Crime e Nichos Culturais

* Nas séries, notamos uma presença muito forte do gênero Crime e de classificações por idioma/origem, o que não aparece com tanto destaque nos filmes:

    * Crime TV Shows aparece três vezes no Top 15 de séries, muitas vezes ligado a dramas ou conteúdo internacional.

    * Aparecem categorias específicas como Spanish-Language, Korean TV Shows, British TV Shows e Anime.

    * Insight: Enquanto os filmes seguem combinações mais tradicionais (Comédia/Drama/Romance), as séries são usadas para explorar nichos culturais e geográficos específicos (como K-Dramas ou produções latinas), que possuem fãs extremamente fiéis.

3. Hibridismo de Gêneros em Filmes

* Nos filmes, o catálogo aposta em combinações que buscam atingir vários sentimentos ao mesmo tempo:

    * Comédias e Dramas (Dramedies) e Dramas e Filmes Independentes são combinações fortes.

    * A tríade Comédia, Drama e Romance aparece em várias permutações entre os títulos mais comuns.

4. Conteúdo Infantil e Séries de Documentários

* Curiosamente, a combinação "Children & Family Movies & Comedies" aparece no Top 15 de filmes, mas em séries, o foco infantil é mais voltado para "Kids' TV & TV Comedies".

    * As "Docuseries" ganham destaque nas séries quando misturadas com Crime ou Internacional, reforçando a tendência de True Crime e documentários seriados.

In [18]:
# --- 1. PROCESSAMENTO DO TEXTO ---
# Usando os títulos do seu DataFrame (substitua 'df_netflix' pelo nome do seu DF, se diferente)
# Concatenamos todos os títulos em uma única string
text = " ".join(df_netflix['title'].dropna().astype(str).tolist())

# Limpeza opcional (embora a WordCloud cuide de muito disso):
# text = text.replace(',', '').replace('[', '').replace("'", '').replace(']', '').replace('.', '')

# --- 2. PREPARAÇÃO DA MÁSCARA ---
# Carregar a imagem fornecida como máscara
try:
    # Abre a imagem fornecida (neste contexto, usamos um espaço reservado, substitua pelo seu arquivo)
    mask_image = Image.open('./netflix_mask.jpg')

    # É crucial converter para escala de cinza e depois um array numpy
    # para que a WordCloud interprete as áreas brancas como transparentes
    mask = np.array(mask_image.convert('L'))
except FileNotFoundError:
    print("Aviso: Arquivo de máscara 'netflix_n.png' não encontrado. Gerando nuvem retangular padrão.")
    mask = None

# --- 3. DEFINIÇÃO DO MAPA DE CORES E STOPWORDS ---
# Mapa de cores personalizado (Paleta Netflix)
cmap = matplotlib.colors.LinearSegmentedColormap.from_list("", [COLORS['Dark_Grey'], COLORS['Netflix_Red_']])

# Adicionamos algumas stopwords (palavras a ignorar) extras se necessário
custom_stopwords = set(STOPWORDS)
# Exemplo: custom_stopwords.update(["The", "A", "Of", "And"]) # se houver mistura de idiomas

# --- 4. GERAÇÃO DA WORDCLOUD (Motor) ---
# Criamos o objeto WordCloud com a máscara do 'N'
wc = WordCloud(
    background_color='white',
    mask=mask,
    colormap=cmap,
    max_words=50,
    width=1000,
    height=1000,
    stopwords=custom_stopwords,
    mode='RGB' # Garante que as cores funcionem bem com a conversão posterior
).generate(text)

# --- 5. INTEGRAÇÃO COM PLOTLY ---
# O Plotly não renderiza a WordCloud diretamente, então converti em uma imagem
# para ser usada como o trace do gráfico.

fig12 = go.Figure()

# Adicionamos a imagem gerada pela WordCloud como um trace do tipo go.Image
# wc.to_array() converte a nuvem de palavras em um array numpy de pixels
fig12.add_trace(go.Image(z=wc.to_array()))

# --- 6. AJUSTES DE LAYOUT ---
# Configuramos o título e limpamos os eixos para o visual de dashboard
fig12.update_layout(
    title=dict(
        text="<b>Palavras mais Comuns nos Títulos do Catálogo</b><br><sup>Nuvem de termos em formato do 'N' da Netflix</sup>",
        x=0.05,
        font=dict(family="serif", size=22, color=COLORS['Dark_Grey'])
    ),
    # Removemos os eixos para parecer uma imagem limpa e não um gráfico matemático
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    plot_bgcolor='white',
    paper_bgcolor='white',
    margin=dict(t=100, b=20, l=50, r=50),
    height=600, # Altura boa para dashboard
    width=600 # Largura quadrada para caber bem o 'N'
)

# Exibir o gráfico interativo do Plotly
fig12.show()

#### ANÁLISE: 

Esta Nuvem de Palavras (Word Cloud), apresentada no formato do "N" da Netflix, revela a semântica e os arquétipos que dominam os títulos do catálogo. É uma análise fascinante que traduz o "sentimento" das obras em termos visuais.

Aqui estão os insights desse gráfico:

1. Temas Universais: O Triângulo "Love, Life, Man"

    * As palavras com maior destaque (maior frequência) são "Love", "Life", "Man" e "World".
        
        * **Análise:** Isso indica que o catálogo é profundamente focado em experiências humanas universais e narrativas de vida. O uso massivo de "Love" e "Life" sugere uma grande oferta de dramas, romances e * histórias biográficas.
        * **Insight:** Títulos que usam esses termos tendem a ter um apelo emocional imediato e são facilmente compreendidos em diferentes culturas (globalização).

2. Identidade e Gênero

    * Palavras como "Girl", "Boy", "Man" e "Woman" aparecem com relevância.

        * **Análise:** O catálogo utiliza arquétipos de gênero para definir suas histórias. Nota-se que "Girl" e "Man" parecem ter um destaque ligeiramente maior que seus pares.
        * **Insight:** Isso reforça a segmentação por público-alvo (Adultos, Teens e Kids) que vimos nos gráficos anteriores, usando a identidade do protagonista diretamente no título para atrair a audiência certa.

3. Conteúdo Sazonal e Temático

    * O termo "Christmas" possui um tamanho considerável na nuvem.

        * **Análise:** Isso revela a importância do conteúdo sazonal para a plataforma. Filmes de Natal são ativos de retenção recorrente todos os anos.
        * **Insight:** A presença de palavras como "Secret", "Story", "Last" e "First" aponta para fórmulas clássicas de narrativa (mistério e jornadas heroicas).

4. Marcas e Localização

    * Palavras como "American", "Legend" e até referências específicas como "Power Rangers" aparecem na nuvem.

        * **Análise:** "American" reforça a origem de grande parte do catálogo (vimos que os EUA detêm 45,9%), enquanto "Legend" e "Story" sugerem um volume alto de épicos ou contos.


# Conclusão e Implicações Estratégicas

Esta análise exploratória, fundamentada em um rigoroso processo de tratamento e visualização de dados, permitiu converter metadados de catálogo em uma visão clara da inteligência de negócio por trás da plataforma.
Os achados validam a hipótese de que o acervo não é apenas volumoso, mas milimetricamente segmentado para equilibrar escala global e fidelização local.

Principais Conclusões:

   **O Binômio Volume vs. Retenção:** Identificamos que, enquanto os Filmes (69,7%) sustentam o volume e a descoberta (atração), as Séries (30,3%) são o motor de retenção. Isso é evidenciado no público infantil, onde as séries superam os filmes, refletindo uma estratégia de engajamento contínuo para essa faixa etária.
    
   **Hegemonia do Drama Internacional:** A combinação de Dramas & International consolidou-se como a espinha dorsal do catálogo em ambos os formatos. Aliado ao fato de que 54,1% do conteúdo já provém de fora dos EUA, fica clara a transição de uma plataforma americana para um ecossistema de entretenimento global e multicultural.

   **Segmentação por Arquétipos:** A análise semântica revelou que o catálogo é "vendido" através de conceitos universais como "Love", "Life" e "World". Esses termos conectam-se diretamente com as preferências mapeadas: o público adulto busca profundidade (Dramas/Independentes) e o adolescente busca conexão emocional (Romance/Comédia).

Implicações Estratégicas:

   **Otimização de Licenciamento:** A queda na produção dos EUA pós-2019, somada à ascensão da Índia, indica uma oportunidade para diversificar fornecedores de dados e conteúdo, reduzindo a dependência de um único polo e mitigando riscos de mercado.

   **Personalização Baseada em Adjacência:** A forte correlação entre gêneros (como Crime e Séries Internacionais) sugere que algoritmos de recomendação podem ser otimizados não apenas pelo que o usuário assiste, mas pela combinação de "tags" que geram maior tempo de tela (watchtime).

   **Sustentabilidade do Modelo:** A predominância de conteúdos para Adultos e Teens posiciona a plataforma como um serviço maduro. Para manter a competitividade, a análise contínua desses padrões de consumo é vital para prever tendências e ajustar a produção original antes mesmo da saturação do gênero.

---

# Referências e Inspirações
*   Documentação Oficial do Pandas, NumPy, Matplotlib, Seaborn, Plotly.
*   Kaggle Dataset: Netflix Movies and TV Shows.
*   Referência Estética e Metodológica: Inspirado nas técnicas de visualização e estrutura de análise de dados de Joshua Swords (Kaggle), adaptado e expandido para os critérios de análise técnica e pré-processamento exigidos nesta disciplina.
